# File 01/2 – XML to DataFrame

### DESCRIPTION: 

#### INPUT FILES:
- ../source_data/treebank-releases-20180919/ # xml files (afnik.xml usw. )
- ../source_data/translations/orv.xml # Translations for the Old East Slavic words
- ../source_data/translations/chu.xml # Translations for the (Old) Church Slavonic words

#### OUTPUT FILES:
- ./OUTPUTS/dataframe_01_02.csv'

## Import Libraries

In [1]:
# ---standard library imports---
import os
import glob
import shutil
from pathlib import Path
from typing import List, Tuple
import xml.etree.ElementTree as ET

# ---third-party imports---
import pandas as pd
# shows status bar for processed XML files
from tqdm import tqdm 

## Delete old output dir and recreate it 

In [2]:
output_dir = "./OUTPUTS"

# remove existing output directory to ensure a clean state
if os.path.isdir(output_dir):
    shutil.rmtree(output_dir)

In [3]:
# create output directory
os.makedirs(output_dir, exist_ok=True)

## Define translation files' paths

In [4]:
# paths for translation files 
TRANSLATION_FILES = {
    "orv": "../source_data/translations/orv.xml", # Old East Slavic 
    "chu": "../source_data/translations/chu.xml" # (Old) Church Slavonic
}

## Corpus Data Model
Define a hierarchical representation of the PROIEL corpus data:

- `Token`: stores linguistic annotations at word level (form, lemma, POS, morphology, dependencies)
- `Sentence`: groups tokens into syntactic units
- `Text`: aggregates sentences and encodes metadata (title, language)

Use `Token`, `Sentence`, and `Text` classes to organize corpus data prior to flattening it into a DataFrame.  


In [5]:
class Token:
    def __init__(self, token_id: str, form: str, lemma: str, pos: str,
                 morphology: str, head_id: str, relation: str, presentation_after: str):
        """
        Represent a token from a PROIEL XML file with its linguistic annotations.

        Args:
            token_id (str): Unique identifier of the token.
            form (str): Surface form of the token.
            lemma (str): Lemma of the form.
            pos (str): Part-of-speech tag.
            morphology (str): Morphological annotation.
            head_id (str): Identifier of the syntactic head token.
            relation (str): Syntactic relation to the head.
            presentation_after (str): Whitespace or punctuation following the token.
        """
        self.token_id = token_id
        self.form = form
        self.lemma = lemma
        self.part_of_speech = pos
        self.morphology = morphology
        self.head_id = head_id
        self.relation = relation
        self.presentation_after = presentation_after

class Sentence:
    def __init__(self, sentence_id: str, tokens: List[Token]):
        """
        Represent a sentence consisting of a unique identifier and a list of tokens.

        Args:
            sentence_id (str): Unique identifier of the sentence.
            tokens (List[Token]): List of Token objects belonging to the sentence.
        """
        self.sentence_id = sentence_id
        self.tokens = tokens

class Text:
    def __init__(self, title: str, sentences: List[Sentence], language: str):
        """
        Represent a text consisting of a title, a list of sentences, and a language code.

        Args:
            title (str): Title of the text.
            sentences (List[Sentence]): List of Sentence objects belonging to the text.
            language (str): Language code of the text.
        """
        self.title = title
        self.sentences = sentences
        self.language = language

## Pipeline
1. **Parsing XML → structured objects**  
   - fn: `parse_proiel_xml_with_language()`  
   - returns: list of `Text` objects (with sentences + tokens) and language code (`orv`, `chu`)<br><br>

2. **Build translation dictionary (core logic)**  
   - fn: `build_translation_dict()`  
   - extracts (lemma, POS) → (ru, en gloss)<br><br>

3. **Load translation file (I/O wrapper)**  
   - fn: `load_translation_file()`  
   - reads XML and applies `build_translation_dict()`<br><br>

4. **Convert to DataFrame**  
   - fn: `texts_to_df()`  
   - flattens structured objects into tabular format

In [6]:
def parse_proiel_xml_with_language(file_path: str) -> Tuple[List[Text], str]:
    """
    Parse a PROIEL XML file and extract structured text data together with the language.

    Args:
        file_path (str): Path to a PROIEL XML file.

    Returns:
        Tuple[List[Text], str]:
            - List[Text]: Parsed texts, each containing sentences and tokens.
            - str: Language code extracted from the <source> element.

    Raises:
        ValueError: If the <source> element is missing in the XML file.
    """

    # parse XML file and obtain root element
    tree = ET.parse(file_path)
    root = tree.getroot()

    # locate <source> element containing metadata and text structure
    source_element = root.find("source")
    # ensure <source> element exists
    if source_element is None:
        raise ValueError("<source> element not found in XML file!")

    # Extract the languge from the <source> attribute
    language = source_element.attrib.get("language", "unknown")

    texts = [] # stores all texts contained in the <source> element
    
    # iterate over <div> elements representing individual texts
    for div in source_element.findall("div"):
        # extract title of the text (fallback "Untitled" if missing)
        title = div.find("title").text if div.find("title") is not None else "Untitled"
        sentences = []

        # iterate over <sentence> elements
        for sentence_elem in div.findall("sentence"):
            # extract sentence identifier
            sentence_id = sentence_elem.attrib.get("id", "Unknown")

            # collect tokens belonging to the current sentence
            tokens = []

            # iterate over <token> elements and extract linguistic annotations
            for token_elem in sentence_elem.findall("token"):
                tokens.append(Token(
                    token_elem.attrib.get("id", "Unknown"),
                    token_elem.attrib.get("form", ""),
                    token_elem.attrib.get("lemma", ""),
                    token_elem.attrib.get("part-of-speech", ""),
                    token_elem.attrib.get("morphology", ""),
                    token_elem.attrib.get("head-id", ""),
                    token_elem.attrib.get("relation", ""),
                    token_elem.attrib.get("presentation-after", "")
                ))
                
            # create Sentence object and add it to current text
            sentences.append(Sentence(sentence_id, tokens))

        # create Text object and add it to result list
        texts.append(Text(title, sentences, language))
        
    # return parsed texts (List[Text] with sentences and tokens) and detected language code
    return texts, language
    

def build_translation_dict(translation_root: ET.Element) -> dict:
    """
    Build a translation dictionary from a translation XML tree.

    The dictionary maps (lemma, part-of-speech) pairs to corresponding
    Russian and English glosses extracted from <gloss> elements.

    Args:
        translation_root (ET.Element): Root element of a translation XML file.

    Returns:
        dict:
            Keys: Tuple[str, str] -> (lemma, part-of-speech)
            Values: Tuple[str, str] -> (Russian gloss, English gloss), 
            where missing glosses are represented as empty strings.
    """

    translation_dict = {}

    # iterate over all lemma elements with a 'lemma' attribute
    for lemma in translation_root.findall(".//lemma[@lemma]"):
        lemma_word = lemma.attrib.get('lemma', '')
        pos = lemma.attrib.get('part-of-speech', '')
        key = (lemma_word, pos)

        # locate glosses element containing translation entries
        glosses = lemma.find('glosses')

        # initialize gloss values (empty if not present)
        russian_gloss = ""
        english_gloss = ""

        # extract glosses for Russian ('rus') and English ('eng')
        if glosses is not None:
            for gloss in glosses.findall('gloss'):
                lang = gloss.attrib.get('language')
                # empty string if no translation can be found
                if lang == "rus":
                    russian_gloss = gloss.text or ""
                elif lang == "eng":
                    english_gloss = gloss.text or ""

        # store translations in dictionary
        translation_dict[key] = (russian_gloss, english_gloss)

    # return translation dictionary mapping (lemma, POS) to glosses    
    return translation_dict


def load_translation_file(language: str) -> dict:
    """
    Load a translation XML file for a given language and convert it into a
    translation dictionary. The XML file is parsed and transformed into
    a dictionary using fn build_translation_dict().

    Args:
        language (str): Language code used to select the translation file.

    Returns:
        dict:
            Translation dictionary mapping (lemma, part-of-speech) to
            (Russian gloss, English gloss).

    Raises:
        ValueError: If no valid translation file exists for the given language.
    """
    # retrieve file path for the given language
    translation_file = TRANSLATION_FILES.get(language)
    # ensure translation file exists
    if not translation_file or not os.path.exists(translation_file):
        raise ValueError(f"No valid translation file for language '{language}' was found!")

    # parse translation XML file
    tree = ET.parse(translation_file)
    root = tree.getroot()

    # build and return translation dictionary
    return build_translation_dict(root)


def texts_to_df(texts: List[Text], translation_dict: dict, file_stem: str) -> pd.DataFrame:
    """
    Convert parsed Text objects into a pandas DataFrame enriched with translation data.

    Args:
        texts (List[Text]): Parsed texts, each containing Sentence and Token objects.
        translation_dict (dict): Mapping from (lemma, part-of-speech) to
            (Russian gloss, English gloss).
        file_stem (str): Base name of the processed file (without extension).

    Returns:
        pd.DataFrame:
            DataFrame where each row represents a token and includes:
            - token-level linguistic annotations
            - sentence- and text-level metadata
            - Russian and English translations (empty strings if not available)
    """
    data = []

    # iterate over all texts, sentences, and tokens to flatten hierarchical structure
    for text in texts:
        for sentence in text.sentences:
            for token in sentence.tokens:

                # build lookup key for translation dictionary
                key = (token.lemma, token.part_of_speech)

                # retrieve translations (default to empty strings if missing)
                russian_gloss, english_gloss = translation_dict.get(key, ("", ""))

                # append token-level record with metadata and translations
                data.append({
                    "File": file_stem,
                    "Text Title": text.title,
                    "Language": text.language,
                    "Sentence ID": sentence.sentence_id,
                    "Token ID": token.token_id,
                    "Form": token.form,
                    "Lemma": token.lemma,
                    "POS": token.part_of_speech,
                    "Morphology": token.morphology,
                    "Head ID": token.head_id,
                    "Relation": token.relation,
                    "Presentation After": token.presentation_after,
                    "Russian Translation": russian_gloss,
                    "English Translation": english_gloss
                })

    # convert collected records into pandas DataFrame
    return pd.DataFrame(data)

## Execution Pipeline
Run the end-to-end processing workflow:
1. **File iteration**  
   Loop over all PROIEL XML files in the input directory.
2. **Parsing**  
   `parse_proiel_xml_with_language()`  
   → converts raw XML into structured objects (`Text`, `Sentence`, `Token`) and detects language.
3. **Translation loading**  
   `load_translation_file()`  
   → loads language-specific gloss mappings (lemma, POS → translations).
4. **DataFrame construction**  
   `texts_to_df()`  
   → flattens structured objects into a tabular format enriched with translations.
5. **Aggregation**  
   → concatenate all intermediate DataFrames into a single dataset (`proiel_df`).

In [7]:
# ---------------------------
# main execution block
# ---------------------------
if __name__ == "__main__":
    # input directory containing PROIEL XML files (Old East Slavic / Old Church Slavonic)
    input_directory = "../source_data/treebank-releases-20180919/"
    xml_files = glob.glob(os.path.join(input_directory, "*.xml"))

    # collect DataFrames for each processed file
    df_list = []

    # iterate over XML files with progress bar
    for file in tqdm(xml_files, desc="Processing XML files"):
        try:
            # parse XML file and extract structured data and language
            texts, language = parse_proiel_xml_with_language(file)
            
            # load corresponding translation dictionary based on language
            if language in TRANSLATION_FILES:
                translation_dict = load_translation_file(language)
            else:
                print(
                    f"Warning: no translation file found for language '{language}'. "
                    "No translations applied."
                )
                translation_dict = {}

            # extract file name without path and extension
            file_stem = os.path.splitext(os.path.basename(file))[0]

            # convert parsed data into DataFrame and store result
            df_temp = texts_to_df(texts, translation_dict, file_stem)
            df_list.append(df_temp)

        # handle parsing errors per file
        except Exception as e:
            print(f"Error processing file '{file}': {e}")

    # concatenate all DataFrames into a single DataFrame
    if df_list:
        proiel_df = pd.concat(df_list, ignore_index=True)
        print(proiel_df)
    else:
        print("No DataFrames to concatenate.")

Processing XML files: 100%|████████████████████████████████████████████████████| 40/40 [00:14<00:00,  2.78it/s]


            File         Text Title Language Sentence ID Token ID        Form  \
0            mst  Mstislav’s letter      orv      189407  2157773          Се   
1            mst  Mstislav’s letter      orv      189407  2157774         азъ   
2            mst  Mstislav’s letter      orv      189407  2157775  мьстиславъ   
3            mst  Mstislav’s letter      orv      189407  2157776  володимирь   
4            mst  Mstislav’s letter      orv      189407  2157777        сн҃ъ   
...          ...                ...      ...         ...      ...         ...   
317277  psal-sin                 54      chu      219251  2339057  нізъведеши   
317278  psal-sin                 54      chu      219251  2339058           ѩ   
317279  psal-sin                 54      chu      219251  2339059          въ   
317280  psal-sin                 54      chu      219251  2339060   стѹденецъ   
317281  psal-sin                 54      chu      219251  2339061   истълѣньѣ   

             Lemma POS  Mor

### Show DataFrame and line of columns

In [8]:
# show the first 5 rows of the newly created data frame
display(proiel_df[:5])

print(f"Number of rows: {proiel_df.shape[0]}")

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,POS,Morphology,Head ID,Relation,Presentation After,Russian Translation,English Translation
0,mst,Mstislav’s letter,orv,189407,2157773,Се,се,I-,---------n,,voc,,"вот, это","behold, here is"
1,mst,Mstislav’s letter,orv,189407,2157774,азъ,азъ,Pp,1s---mn--i,2157784,sub,,я,I
2,mst,Mstislav’s letter,orv,189407,2157775,мьстиславъ,мьстиславъ,Ne,-s---mn--i,2157774,apos,,Мстислав,Mstislav
3,mst,Mstislav’s letter,orv,189407,2157776,володимирь,володимирь,A-,-s---mnpsi,2157777,atr,,Владимира,Vladimir's
4,mst,Mstislav’s letter,orv,189407,2157777,сн҃ъ,сынъ,Nb,-s---mn--i,2157775,apos,,сын,son


Number of rows: 317282


### Add column "Type" to the DataFrame
- **"Type"** indicates the language group:
- **Abbreviations**:
  - *OCS*: Old Church Slavonic
  -  *CS*: Church Slavonic
  -  *OR*: Old Russian (syn.: Old East Slavic)

In [9]:
type_mapping = {
    "OCS": ["supr", "zogr", "kiev-mis", "psal-sin"],
    "CS": ["vit-const", "vit-meth"],
    "OR": [
        "afnik", "birchbark", "smol-pol-lit", "mstislav-col", "ostromir-col", "peter", "domo", "sergrad",
        "schism", "pskov-ivan", "rig-smol1281", "mst", "nov-marg", "novgorod-jaroslav", "rusprav",
        "ust-vlad", "dux-grjaz", "riga-goth", "nov-sin", "kiev-hyp", "avv", "nov-list", "pvl-hyp",
        "lav", "suz-lav", "drac", "spi", "luk-koloc", "pskov", "const", "usp-sbor", "varlaam",
        "vest-kur", "zadon"
    ]
}

# assign language group label ("Type") based on file identifier
# unknown identifiers are labeled as "UNK"
def get_type(file_id: str) -> str:
    """
    Map a file identifier to its corresponding language group.

    Args:
        file_id (str): Identifier from the "File" column.

    Returns:
        str:
            Language group label ("OCS", "CS", "OR"), or "UNK" if the identifier
            is not found in the mapping.
    """
    for lang, ids in type_mapping.items():
        if file_id in ids:
            return lang
    return "UNK"


# apply mapping to fn "get_type()" to create new column "Type"
proiel_df["Type"] = proiel_df["File"].apply(get_type)

## Show dataframe example and info

In [10]:
# display first row to verify structure and content
display(proiel_df[:1])
# display DataFrame schema (column names, data types, non-null counts)
proiel_df.info()

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,POS,Morphology,Head ID,Relation,Presentation After,Russian Translation,English Translation,Type
0,mst,Mstislav’s letter,orv,189407,2157773,Се,се,I-,---------n,,voc,,"вот, это","behold, here is",OR


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 317282 entries, 0 to 317281
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   File                 317282 non-null  object
 1   Text Title           317282 non-null  object
 2   Language             317282 non-null  object
 3   Sentence ID          317282 non-null  object
 4   Token ID             317282 non-null  object
 5   Form                 317282 non-null  object
 6   Lemma                317282 non-null  object
 7   POS                  317282 non-null  object
 8   Morphology           317282 non-null  object
 9   Head ID              317282 non-null  object
 10  Relation             317282 non-null  object
 11  Presentation After   317282 non-null  object
 12  Russian Translation  317282 non-null  object
 13  English Translation  317282 non-null  object
 14  Type                 317282 non-null  object
dtypes: object(15)
memory usage: 36.3+ 

In [11]:
# show a list of the unique files contained in column "File"
unique_files         = proiel_df["File"].unique()
print(unique_files)
# show sum of unique files in the data frame
print(len(unique_files))

['mst' 'mstislav-col' 'supr' 'vit-meth' 'birchbark' 'pskov' 'const'
 'luk-koloc' 'lav' 'smol-pol-lit' 'nov-sin' 'avv' 'kiev-hyp' 'peter'
 'vest-kur' 'spi' 'zadon' 'rusprav' 'pskov-ivan' 'rig-smol1281' 'zogr'
 'drac' 'sergrad' 'nov-list' 'kiev-mis' 'ostromir-col' 'varlaam' 'afnik'
 'dux-grjaz' 'vit-const' 'ust-vlad' 'riga-goth' 'domo' 'usp-sbor' 'schism'
 'nov-marg' 'suz-lav' 'novgorod-jaroslav' 'pvl-hyp' 'psal-sin']
40


## Write dataframe to file

In [12]:
# write the data frame to csv
output_file_path = os.path.join(output_dir, "dataframe_01_02.csv")
proiel_df.to_csv(output_file_path, index=False)

## Validation Checks

In [13]:
# ASSERT:
## data frame contains 40 files  
assert len(unique_files) == 40, "Unexpected number of input files"
## csv was successfully written to the 
assert Path(output_file_path).exists(), "Error creating output file"